# Student Performance Prediction System
### CODEVEDX Internship - Task 2: Training + Data Analysis Tool

**Objective:**
Develop an end-to-end Machine Learning model that analyzes student academic records and predicts final examination performance using attendance percentage, previous examination marks, and daily study hours.

## 1. Environment Setup & Data Ingestion

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
sns.set_theme(style="darkgrid")

# Load dataset
data_path = Path("../data/raw/student_data.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Descriptive Statistics & Data Health

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nSummary Statistics:")
df.describe().T

## 3. Correlation Heatmap

In [ ]:
plt.figure(figsize=(9, 6))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap="Purples", fmt=".2f", cbar=True)
plt.title("Academic Correlation Matrix (Pearson)")
plt.tight_layout()
plt.show()

## 4. Key Relationships (Study Hours & Attendance vs Final Marks)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.regplot(data=df, x="daily_study_hours", y="final_exam_marks", ax=axes[0], color="#6366F1")
axes[0].set_title("Daily Study Hours vs Final Marks")

sns.regplot(data=df, x="attendance_percentage", y="final_exam_marks", ax=axes[1], color="#06B6D4")
axes[1].set_title("Attendance % vs Final Marks")

plt.tight_layout()
plt.show()

## 5. Baseline Machine Learning Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error

features = ["attendance_percentage", "previous_exam_marks", "daily_study_hours", "assignments_completed", "sleep_hours"]
X = df[features]
y = df["final_exam_marks"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    r2 = r2_score(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    results.append({"Model": name, "R2": round(r2, 4), "RMSE": round(rmse, 2), "MAE": round(mae, 2)})

pd.DataFrame(results).sort_values(by="R2", ascending=False)